<a href="https://colab.research.google.com/github/mirza-hannan-baber/Sales-Intelligence-Dashboard/blob/main/Week1_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Sales Intelligence Dashboard - Week 1
## Task: Data Generation, Noise Injection & Data Cleaning

**Objectives:**
- Generate realistic B2B CRM sales dataset (6000 deals across 3+ years).
- Inject realistic dataset noise (missing values, extreme outliers, inconsistent formatting).
- Perform data cleaning, handle nulls, filter invalid outliers, and standardize categorical variables.


In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_rows = 5000

# 1. Base Setup
industries = ['IT & Tech', 'Healthcare', 'Finance', 'Retail', 'Manufacturing']
ind_probs = [0.30, 0.20, 0.20, 0.15, 0.15]

opportunity_ids = [f'OPP-{1000+i}' for i in range(n_rows)]
selected_industries = np.random.choice(industries, n_rows, p=ind_probs)

# 2. Industry-Based Deal Size Patterns (Finance/Healthcare = Expensive)
deal_sizes = []
for ind in selected_industries:
    if ind in ['Finance', 'Healthcare']:
        # High Value Deals ($40k to $200k+)
        val = np.random.lognormal(mean=11.2, sigma=0.6)
    elif ind in ['IT & Tech', 'Manufacturing']:
        # Mid-to-High Value Deals ($20k to $100k)
        val = np.random.lognormal(mean=10.5, sigma=0.5)
    else: # Retail
        # Small-to-Mid Value Deals ($5k to $35k)
        val = np.random.lognormal(mean=9.5, sigma=0.5)
    deal_sizes.append(round(val, 2))

deal_sizes = np.array(deal_sizes)

# 3. Correlation Pattern: Deal Size vs Sales Cycle (Bari deal = Lambi Duration)
# Base days + multiplier based on deal size log value
sales_cycles = (np.log(deal_sizes) * 7 + np.random.normal(loc=0, scale=12, size=n_rows)).astype(int)
sales_cycles = np.clip(sales_cycles, 15, 180) # Normal range between 15 and 180 days

# 4. Win Rate Pattern (Choti deals win jaldi hoti hain, bari deals lose zyada hoti hain)
win_probs = 1 / (1 + np.exp((deal_sizes - np.median(deal_sizes)) / 30000)) # Sigmoid curve
statuses = ['Won' if np.random.rand() < p else 'Lost' for p in win_probs]

# Base Clean Data Frame
df_raw = pd.DataFrame({
    'Opportunity_ID': opportunity_ids,
    'Industry': selected_industries,
    'Deal_Size': deal_sizes,
    'Sales_Cycle_Days': sales_cycles,
    'Status': statuses,
    'Close_Date': pd.date_range(start='2023-01-01', end='2026-07-30', periods=n_rows)
})


# Noise 1: Missing Values (Nulls)
df_raw.loc[df_raw.sample(frac=0.04, random_state=1).index, 'Deal_Size'] = np.nan
df_raw.loc[df_raw.sample(frac=0.03, random_state=2).index, 'Sales_Cycle_Days'] = np.nan

# Noise 2: Extreme Outliers
df_raw.loc[df_raw.sample(n=8, random_state=3).index, 'Deal_Size'] = 18000000.00  # $18M Impossible Outlier
df_raw.loc[df_raw.sample(n=5, random_state=4).index, 'Sales_Cycle_Days'] = -30     # Negative Days Error

# Noise 3: Categorical Formatting Errors
df_raw['Industry'] = df_raw['Industry'].replace({
    'Healthcare': 'health-care',
    'IT & Tech': 'IT_Tech'
})

# 💾 Save DIRTY DATASET First!
df_raw.to_csv('dirty_crm_sales_data.csv', index=False)
print("✅ Pattern-Based Dirty Dataset Saved as 'dirty_crm_sales_data.csv'!")
print("Length of the dataset is : ", len(df_raw))
df_raw.head(10)

✅ Pattern-Based Dirty Dataset Saved as 'dirty_crm_sales_data.csv'!
Length of the dataset is :  5000


,Opportunity_ID,Industry,Deal_Size,Sales_Cycle_Days,Status,Close_Date
0,OPP-1000,health-care,51114.09,74.0,Won,2023-01-01 00:00:00.000000000
1,OPP-1001,Manufacturing,10991.17,69.0,Lost,2023-01-01 06:16:12.194438887
2,OPP-1002,Retail,10871.39,53.0,Won,2023-01-01 12:32:24.388877775
3,OPP-1003,Finance,126510.96,80.0,Lost,2023-01-01 18:48:36.583316663
4,OPP-1004,IT_Tech,47515.68,72.0,Lost,2023-01-02 01:04:48.777755551
5,OPP-1005,IT_Tech,44985.39,77.0,Won,2023-01-02 07:21:00.972194438
6,OPP-1006,IT_Tech,31571.29,76.0,Lost,2023-01-02 13:37:13.166633326
7,OPP-1007,Manufacturing,19158.17,75.0,Lost,2023-01-02 19:53:25.361072214
8,OPP-1008,Finance,99625.81,83.0,Lost,2023-01-03 02:09:37.555511102
9,OPP-1009,Retail,8803.10,49.0,Won,2023-01-03 08:25:49.749949990


## Section 2: Data Inspection & Noise Identification

In [6]:
# Dirty Data ki copy banayein inspection aur cleaning ke liye
df_clean = df_raw.copy()

print("=== 1. Missing Values (Nulls) Check ===")
print(df_clean.isnull().sum())

print("\n=== 2. Industry Column Unique Values (Formatting Issues) ===")
print(df_clean['Industry'].value_counts())

print("\n=== 3. Summary Statistics (Outlier Spotting) ===")
print(df_clean[['Deal_Size', 'Sales_Cycle_Days']].describe().round(2))

=== 1. Missing Values (Nulls) Check ===
Opportunity_ID        0
Industry              0
Deal_Size           199
Sales_Cycle_Days    150
Status                0
Close_Date            0
dtype: int64

=== 2. Industry Column Unique Values (Formatting Issues) ===
Industry
IT_Tech          1531
Finance          1030
health-care       969
Retail            740
Manufacturing     730
Name: count, dtype: int64

=== 3. Summary Statistics (Outlier Spotting) ===
         Deal_Size  Sales_Cycle_Days
count      4801.00           4850.00
mean      85817.91             73.78
std      733590.99             13.57
min        2641.87            -30.00
25%       24983.03             65.00
50%       42581.48             74.00
75%       70542.98             83.00
max    18000000.00            118.00
